<a href="https://colab.research.google.com/github/irfkirf/Norm-dtsc-3601-project/blob/main/Untitled3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# =========================
# 0) Setup
# =========================
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.metrics import precision_recall_fscore_support

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# File path to your cleaned dataset from Sprint 2
DATA_PATH = "cleaned_property_crimes.csv"  # change if needed
RANDOM_STATE = 42


In [2]:
# =========================
# 1) Load data
# =========================
df = pd.read_csv(DATA_PATH)

# Expecting columns similar to your screenshots:
# 'crime_group', 'crime_binary', 'Vict Age Group', 'Vict Sex', 'Vict Descent',
# 'AREA', 'LAT', 'LON', 'Hour', 'Month', 'Day0fWeek', (optionally 'Premis Cd')

print("Rows, Cols:", df.shape)
df.head()


Rows, Cols: (411206, 15)


,crime_group,crime_binary,Crm Cd,Crm Cd Desc,Vict Age,Vict Age Group,Vict Sex,Vict Descent,AREA,AREA NAME,Premis Cd,Premis Desc,LAT,LON,DATE OCC
0,Theft,0,354,THEFT OF IDENTITY,31.0,31-50,M,H,15,N Hollywood,501.0,SINGLE FAMILY DWELLING,34.2124,-118.4092,11/07/2020 12:00:00 AM
1,Theft,0,354,THEFT OF IDENTITY,30.0,18-30,M,W,9,Van Nuys,501.0,SINGLE FAMILY DWELLING,34.1847,-118.4509,10/30/2020 12:00:00 AM
2,Theft,0,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND ...,47.0,31-50,F,A,7,Wilshire,101.0,STREET,34.0339,-118.3747,12/24/2020 12:00:00 AM
3,Theft,0,420,THEFT FROM MOTOR VEHICLE - PETTY ($950 & UNDER),63.0,51+,M,H,14,Pacific,103.0,ALLEY,33.9813,-118.4350,09/29/2020 12:00:00 AM
4,Theft,0,354,THEFT OF IDENTITY,35.0,31-50,M,B,4,Hollenbeck,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",34.0830,-118.1678,11/11/2020 12:00:00 AM


In [3]:
# =========================
# 2) Define target + features
# =========================
# Binary target (from your notebook): 1 = Theft, 0 = Break-in/Burglary
target_col = "crime_binary"

# Choose low-cardinality, useful features you already explored
categorical_cols = [
    "Vict Age Group", "Vict Sex", "Vict Descent",
    "AREA", "Month", "Day0fWeek"
]
numeric_cols = ["Hour", "LAT", "LON"]

# Keep only rows with non-missing target
df = df.dropna(subset=[target_col])

# Ensure all chosen columns exist
available_cat = [c for c in categorical_cols if c in df.columns]
available_num = [c for c in numeric_cols if c in df.columns]
X = df[available_cat + available_num].copy()
y = df[target_col].astype(int)

print("Categorical features used:", available_cat)
print("Numeric features used:", available_num)
print("Target distribution (1=Theft, 0=Break/Burg):")
print(y.value_counts(normalize=True).round(3))


Categorical features used: ['Vict Age Group', 'Vict Sex', 'Vict Descent', 'AREA']
Numeric features used: ['LAT', 'LON']
Target distribution (1=Theft, 0=Break/Burg):
crime_binary
0    0.693
1    0.307
Name: proportion, dtype: float64


In [4]:
# =========================
# 3) Train/Test split
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y
)
X_train.shape, X_test.shape


((328964, 6), (82242, 6))

In [5]:
# =========================
# 4) Preprocessing
# =========================
# One-hot encode categoricals; standardize numerics for Logistic Regression.
# (Trees don't need scaling, but leaving numerics "passthrough" is fine in the same preprocessor.)

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), available_cat),
        ("num", "passthrough", available_num)
    ]
)


In [6]:
# =========================
# 5) Define models + param grids
#    (keep grids small so they finish fast)
# =========================

# Logistic Regression
pipe_log = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", LogisticRegression(max_iter=500, class_weight="balanced", n_jobs=None))
])
grid_log = {
    "clf__C": [0.5, 1.0, 2.0],
    "clf__penalty": ["l2"],
    "clf__solver": ["lbfgs"]
}

# Decision Tree
pipe_dt = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", DecisionTreeClassifier(random_state=RANDOM_STATE, class_weight="balanced"))
])
grid_dt = {
    "clf__max_depth": [8, 12, 16, None],
    "clf__min_samples_split": [2, 10, 50],
    "clf__min_samples_leaf": [1, 5, 20]
}

# Random Forest
pipe_rf = Pipeline(steps=[
    ("prep", preprocessor),
    ("clf", RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced"))
])
grid_rf = {
    "clf__n_estimators": [150, 250],
    "clf__max_depth": [12, 20, None],
    "clf__min_samples_split": [2, 10],
    "clf__min_samples_leaf": [1, 5]
}

models = [
    ("Logistic Regression", pipe_log, grid_log),
    ("Decision Tree",       pipe_dt,  grid_dt),
    ("Random Forest",       pipe_rf,  grid_rf),
]


In [ ]:
# =========================
# 6) Fit + tune with 3-fold CV, evaluate on held-out test set
# =========================
results = []

for name, pipe, grid in models:
    print(f"\n=== {name} ===")
    gs = GridSearchCV(
        estimator=pipe,
        param_grid=grid,
        scoring="accuracy",
        cv=3,
        n_jobs=-1,
        verbose=1,
    )
    gs.fit(X_train, y_train)

    best_model = gs.best_estimator_
    y_pred = best_model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, y_pred, average="binary", zero_division=0)

    print("Best Params:", gs.best_params_)
    print("Test Accuracy:", round(acc, 4))
    print("\nClassification Report:\n", classification_report(y_test, y_pred, zero_division=0))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

    results.append({
        "Model": name,
        "Test Accuracy": acc,
        "Precision (Theft=1)": prec,
        "Recall (Theft=1)": rec,
        "F1 (Theft=1)": f1,
        "Best Params": gs.best_params_
    })

res_df = pd.DataFrame(results).sort_values("Test Accuracy", ascending=False).reset_index(drop=True)
res_df



=== Logistic Regression ===
Fitting 3 folds for each of 3 candidates, totalling 9 fits


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Best Params: {'clf__C': 0.5, 'clf__penalty': 'l2', 'clf__solver': 'lbfgs'}
Test Accuracy: 0.5477

Classification Report:
               precision    recall  f1-score   support

           0       0.77      0.49      0.60     57033
           1       0.37      0.67      0.48     25209

    accuracy                           0.55     82242
   macro avg       0.57      0.58      0.54     82242
weighted avg       0.65      0.55      0.56     82242

Confusion Matrix:
 [[28153 28880]
 [ 8316 16893]]

=== Decision Tree ===
Fitting 3 folds for each of 36 candidates, totalling 108 fits


In [ ]:
# =========================
# 7) Baseline (Majority class) for comparison
# =========================
baseline_pred = np.ones_like(y_test)  # predict class 1 (Theft) for all
baseline_acc = accuracy_score(y_test, baseline_pred)
print("Baseline (always Theft) Accuracy:", round(baseline_acc, 4))


In [ ]:
# =========================
# 8) Pretty print summary table
# =========================
summary = res_df.copy()
summary["Test Accuracy"] = (summary["Test Accuracy"]*100).round(2).astype(str) + "%"
summary["Precision (Theft=1)"] = summary["Precision (Theft=1)"].round(3)
summary["Recall (Theft=1)"] = summary["Recall (Theft=1)"].round(3)
summary["F1 (Theft=1)"] = summary["F1 (Theft=1)"].round(3)
summary


In [ ]:
# =========================
# 9) Quick takeaway text you can paste into slides
# =========================
best_row = res_df.iloc[0]
print(
    f"Best model: {best_row['Model']} | "
    f"Acc={best_row['Test Accuracy']:.3f}, "
    f"Prec={best_row['Precision (Theft=1)']:.3f}, "
    f"Rec={best_row['Recall (Theft=1)']:.3f}, "
    f"F1={best_row['F1 (Theft=1)']:.3f}\n"
    f"Beats baseline accuracy of ~{baseline_acc:.3f} by "
    f"{(best_row['Test Accuracy']-baseline_acc)*100:.1f} percentage points.\n"
    f"Best Params: {best_row['Best Params']}"
)
